### Library

In [1]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [2]:
import multiprocessing as mp
if mp.get_start_method(allow_none=True) is None:
    mp.set_start_method('spawn')

import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', message='.*input_shape.*')
warnings.filterwarnings('ignore', message='.*structure of.*inputs.*')

import os, time, gc
from types import SimpleNamespace

import numpy as np
import pandas as pd
import time as time_module
from scipy.stats import t
from scipy.special import kv, gamma

import jax, jax.numpy as jnp

import tensorflow as tf
from tensorflow.keras import mixed_precision
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber
from tensorflow.keras import backend as K

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import OneHotEncoder

import optuna
import plotly.io as pio

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature

2026-02-04 10:22:57.789202: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770171777.803132 1743691 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770171777.807106 1743691 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770171777.818238 1743691 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770171777.818268 1743691 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770171777.818270 1743691 computation_placer.cc:177] computation placer alr

### Environment setting

In [3]:
np_f32 = np.float32
jnp_f32 = jnp.float32
dtype_basis = np.float32

jax.config.update("jax_enable_x64", False)

pio.renderers.default = "notebook"
warnings.filterwarnings("ignore", category=UserWarning)

os.environ.update({"TF_CPP_MIN_LOG_LEVEL": "2"})
optuna.logging.set_verbosity(optuna.logging.WARNING)

os.environ.setdefault("OMP_NUM_THREADS", "12")
os.environ.setdefault("MKL_NUM_THREADS", "12")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "12")

def init_hardware(dtype: str = "float32"):
    gpus = tf.config.list_physical_devices("GPU")
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    strategy = (tf.distribute.MirroredStrategy() if len(gpus) > 1 else tf.distribute.get_strategy())
    mixed_precision.set_global_policy(dtype)
    return strategy

strategy = init_hardware(dtype="float32")

### Auto save notebook

In [4]:
from IPython.display import display, Javascript

def save_notebook():
    display(Javascript('IPython.notebook.save_checkpoint()'))
    current_time = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
    print(f"💾 Notebook saved at {current_time}")

### Our function

In [5]:
from spherical_deepkriging.basis_functions.wendland.wenland import wendland
from spherical_deepkriging.basis_functions.mrts.mrts import mrts0

from rpy2.robjects.conversion import localconverter
from spherical_deepkriging.basis_functions.mrts_sphere.sphere import mrts_sphere, numpy2ri_converter

from spherical_deepkriging.models.deep_kriging import DeepKrigingTrainer, DeepKrigingDefaultTrainer
from spherical_deepkriging.configs import DeepKrigingModelConfig, DeepKrigingDefaultConfig
# from spherical_deepkriging.models.universal_kriging import UniversalKriging

R callback write-console: Registered S3 methods overwritten by 'RcppEigen':
  method               from         
  predict.fastLm       RcppArmadillo
  print.fastLm         RcppArmadillo
  summary.fastLm       RcppArmadillo
  print.summary.fastLm RcppArmadillo
  


### Simulation Helper

In [6]:
def simulate_data(num_sample, seed):
    """
    Experiment I : Stationary Gaussian processes + Eggholder Mean Function.
    """
    rng = np.random.default_rng(seed)
    
    # Generate spherical coordinates uniformly
    phi = rng.uniform(0, 2 * np.pi, num_sample)
    theta = np.arccos(rng.uniform(-1, 1, num_sample))
    lat_rad = np.pi/2 - theta
    lon_rad = phi - np.pi
    
    lat_deg = np.rad2deg(lat_rad).astype(np.float32)
    lon_deg = np.rad2deg(lon_rad).astype(np.float32)
    
    # Convert to Cartesian coordinates for distance calculation
    x_cart = np.cos(lat_rad) * np.cos(lon_rad)
    y_cart = np.cos(lat_rad) * np.sin(lon_rad)
    z_cart = np.sin(lat_rad)

    # Eggholder Mean Term
    x = 128.0 * x_cart
    y = 128.0 * y_cart
    
    term1 = -(y + 47.0) * np.sin(np.sqrt(np.abs(x + y/2.0 + 47.0)))
    term2 = -x * np.sin(np.sqrt(np.abs(x - (y + 47.0))))
    mean_term = (term1 + term2).astype(np.float32)
    
    coords = np.column_stack([x_cart, y_cart, z_cart]).astype(np.float32)
    
    # Covariance matrix with exponential covariance function
    dist_matrix = np.sqrt(((coords[:, None, :] - coords[None, :, :]) ** 2).sum(axis=2))
    cov_matrix = np.exp(-dist_matrix / 0.5).astype(np.float32)
    
    # Add jitter for numerical stability (same as old function)
    jitter = 1e-3
    cov_matrix += np.float32(jitter) * np.eye(num_sample, dtype=np.float32)
    
    # Cholesky decomposition with error handling (same as old function)
    try:
        L = np.linalg.cholesky(cov_matrix)
    except np.linalg.LinAlgError:
        cov_matrix += np.float32(1e-2) * np.eye(num_sample, dtype=np.float32)
        try:
            L = np.linalg.cholesky(cov_matrix)
        except np.linalg.LinAlgError:
            eigenvals, eigenvecs = np.linalg.eigh(cov_matrix)
            eigenvals = np.maximum(eigenvals, 1e-6)
            L = eigenvecs @ np.diag(np.sqrt(eigenvals))
    
    # Generate Gaussian process: y = mean + L @ z
    z_std = rng.standard_normal(num_sample).astype(np.float32)
    y = mean_term + (L @ z_std).astype(np.float32)
    
    z = y
    
    print(f"\nSimulate Data | z mean: {np.mean(y):.4f} ({np.std(y) / np.sqrt(num_sample):.4f}), Variance: {np.var(y, ddof=0):.4f}, Range: [{np.min(y):.4f}, {np.max(y):.4f}]")
    
    del phi, theta, lat_rad, lon_rad, x_cart, y_cart, z_cart, coords, dist_matrix, cov_matrix, L, z_std
    gc.collect()
    
    return pd.DataFrame({
        "longitude": lon_deg,
        "latitude": lat_deg,
        "z": z,
    })

### Helper

In [7]:
def data_preprocessing(data_frame):
    data = data_frame.copy()

    numeric_cols = ["longitude", "latitude", "z"]
    data[numeric_cols] = data[numeric_cols].apply(pd.to_numeric, errors="coerce")
    data.dropna(subset=numeric_cols, inplace=True)

    lon, lat = data["longitude"].to_numpy(), data["latitude"].to_numpy()

    norm_lon = (lon - lon.min()) / (lon.max() - lon.min())
    norm_lat = (lat - lat.min()) / (lat.max() - lat.min())

    location_data = np.column_stack([lat, lon]).astype("float32")
    location_data_norm = np.column_stack([norm_lon, norm_lat]).astype("float32")
    y_combined = data['z'].to_numpy().astype("float32")[:, None]

    # Handle
    categorical_data = None

    return location_data, location_data_norm, categorical_data, y_combined


def precompute_wendland(location, num_basis):
    parts = []
    for nb in num_basis:
        grid = np.column_stack(np.meshgrid(
            np.linspace(0, 1, int(np.sqrt(nb)), dtype=np_f32),
            np.linspace(0, 1, int(np.sqrt(nb)), dtype=np_f32),
        )).reshape(-1, 2).astype(np_f32)

        theta = np_f32(2.5)/np_f32(np.sqrt(nb))
        parts.append(
            wendland(location, grid, theta=theta, k=2)
        )

        # Clean up the memory
        del grid
        gc.collect()

    return np.hstack(parts).astype(dtype_basis, copy=False)


def precompute_max_mrts(distance_type, location_data, knot_num, order_max, knot=None):
    if knot is None:
        idx_knot = np.random.choice(location_data.shape[0], knot_num, replace=False)
        knot = location_data[idx_knot].astype(np_f32)
    else:
        idx_knot = None

    if distance_type == "sphere":
        with localconverter(numpy2ri_converter):
            res_r = mrts_sphere(knot, order_max, location_data.astype(np_f32))
        res_dict = dict(zip(res_r.names(), res_r))
        phi = np.asarray(res_dict["mrts"], dtype=dtype_basis)
    else:
        phi = np.asarray(
            mrts0(jnp.asarray(knot, dtype=jnp_f32), k=order_max, 
                  x=jnp.asarray(location_data, dtype=jnp_f32)), dtype=dtype_basis
        )

    return phi, idx_knot, knot


def prepare_data(categorical_data, basis, y_combined, seed, split_ratio=(0.8, 0.1, 0.1)):
    idx_all = np.arange(basis.shape[0])
    train_ratio, val_ratio, test_ratio = split_ratio
    
    train_val_x1, test_x1 = train_test_split(
        idx_all, train_size=train_ratio+val_ratio, random_state=seed)
    train_x1, val_x1 = train_test_split(
        train_val_x1, train_size=train_ratio/(train_ratio+val_ratio), random_state=seed)
    
    X_train_cont, X_val_cont, X_test_cont = (
        basis[train_x1], basis[val_x1], basis[test_x1])
    y_train, y_val, y_test = (
        y_combined[train_x1], y_combined[val_x1], y_combined[test_x1])
    
    def flatten(targets):
        return targets.reshape(-1).astype(np_f32, copy=False)
    y_train_flat, y_val_flat, y_test_flat = map(flatten, [y_train, y_val, y_test])

    def flatten(covariates):
        return covariates.reshape(-1, basis.shape[1]).astype(np_f32)
    X_train_cont_flat, X_val_cont_flat, X_test_cont_flat = map(flatten, [X_train_cont, X_val_cont, X_test_cont])
    
    # Handle categorical features
    if categorical_data is None:
        zero_vector = lambda n: np.zeros((n, 0), dtype=np_f32)
        X_train_cat, X_val_cat, X_test_cat = map(zero_vector, [len(X_train_cont_flat), len(X_val_cont_flat), len(X_test_cont_flat)])
    else:
        cat_train = categorical_data.reshape(-1, 1)[train_x1]
        cat_val = categorical_data.reshape(-1, 1)[val_x1]
        cat_test = categorical_data.reshape(-1, 1)[test_x1]
        
        OHE = OneHotEncoder(sparse_output=False, dtype=np_f32)
        X_train_cat = OHE.fit_transform(cat_train).astype(np_f32)
        X_val_cat = OHE.transform(cat_val).astype(np_f32)
        X_test_cat = OHE.transform(cat_test).astype(np_f32)
    
    return (X_train_cont_flat, X_train_cat, y_train_flat,
            X_val_cont_flat, X_val_cat, y_val_flat,
            X_test_cont_flat, X_test_cat, y_test_flat)


def print_config(name_model, config):
    """Print all configuration parameters."""
    print(f"\n{'='*70}")
    print(f"📋 Model Config for {name_model}")
    print(f"{'='*70}")
    
    # Get all attributes from config
    config_dict = vars(config)
    
    # Print each parameter
    for key, value in sorted(config_dict.items()):
        # Format the value for better readability
        # Check basic types FIRST (important!)
        if value is None:
            value_str = 'None'
        elif isinstance(value, bool):  # Must check bool before int!
            value_str = str(value)
        elif isinstance(value, (int, float)):
            value_str = str(value)
        elif isinstance(value, str):
            value_str = f"'{value}'"
        elif isinstance(value, (list, tuple)):
            value_str = str(value)
        elif hasattr(value, '__name__'):  # For functions/classes
            value_str = value.__name__
        else:  # For complex objects (optimizer, loss, etc.)
            class_name = value.__class__.__name__
            value_str = f"{class_name}(...)"
        
        # Truncate very long strings
        if len(value_str) > 60:
            value_str = value_str[:57] + '...'
        
        print(f"  {key:20s}: {value_str}")
    
    print(f"{'='*70}\n")


def train_eval(name_model, epochs, batch_size, loss, dropout_rate,
               X_train, X_train_cat, y_train,
               X_val, X_val_cat, y_val,
               X_test, X_test_cat, y_test):

    if name_model in ["OLS_wendland", "OLS_sphere"]:
        t0 = time.time()
        model = LinearRegression().fit(X_train, y_train)
            
        val_loss = float(mean_squared_error(y_val, model.predict(X_val)))
        y_pred = model.predict(X_test).astype(np_f32).reshape(-1)
        training_time = time.time() - t0
        
        metrics = {
            "Model": name_model,
            "Val_loss": float(val_loss),
            "MSPE": float(mean_squared_error(y_test, y_pred)),
            "RMSE": float(np.sqrt(float(mean_squared_error(y_test, y_pred)))),
            "MAE": float(mean_absolute_error(y_test, y_pred)),
            "R2": float(r2_score(y_test, y_pred)),
            "Time": float(training_time),
        }
        
        return metrics, model
    
    elif name_model == "DeepKriging_wendland":
        config = DeepKrigingDefaultConfig(
            input_dim=X_train.shape[1],
            output_type='continuous',
            optimizer=Adam(learning_rate=1e-3),
            loss=loss,
            epochs=epochs,
            batch_size=batch_size,
            verbose=0
        )

        print_config(name_model, config)

    elif name_model == "DeepKriging_wendland_new":
        optimizer = Adam(learning_rate=5e-3)
        config = DeepKrigingModelConfig(
            input_dim=X_train.shape[1],
            output_type='continuous',
            hidden_layers=[1024, 512, 256, 128, 64],
            activation='relu',
            dropout_rate=dropout_rate,
            optimizer=optimizer,
            loss=loss,
            metrics=['mae'],
            epochs=epochs,
            batch_size=batch_size,
            patience=40,
            verbose=0
        )

        print_config(name_model, config)

    elif name_model in ["DeepKriging_mrts", "DeepKriging_sphere", "DeepKriging_sphere_Huber"]:
        optimizer = Adam(learning_rate=5e-3)
        config = DeepKrigingModelConfig(
            input_dim=X_train.shape[1],
            output_type='continuous',
            hidden_layers=[1024, 512, 256, 128, 64],
            activation='relu',
            dropout_rate=dropout_rate,
            optimizer=optimizer,
            loss=loss,
            metrics=['mae'],
            epochs=epochs,
            batch_size=batch_size,
            patience=40,
            verbose=0
        )

        print_config(name_model, config)

    t0 = time.time()
    with strategy.scope():
        if name_model == "DeepKriging_wendland":
            model = DeepKrigingDefaultTrainer(config)
        elif name_model == "DeepKriging_wendland_new":
            model = DeepKrigingTrainer(config)
        else:
            model = DeepKrigingTrainer(config)
        model.model.compile(optimizer=config.optimizer, loss=config.loss, metrics=config.metrics)

    checkpoint_path = f"best_{name_model}_{time.time_ns()}.weights.h5"
    if name_model == "DeepKriging_wendland":
        callbacks = [
            tf.keras.callbacks.ModelCheckpoint(
                filepath=checkpoint_path, monitor="val_loss", mode="min",
                save_best_only=True, save_weights_only=True, verbose=0)
        ]
    else:
        callbacks = [
            tf.keras.callbacks.ModelCheckpoint(
                filepath=checkpoint_path, monitor="val_loss", mode="min",
                save_best_only=True, save_weights_only=True, verbose=0),
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=config.patience, restore_best_weights=True, verbose=0),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=max(5, config.patience // 2), min_lr=1e-6, verbose=0)
        ]

    train_dataset = tf.data.Dataset.from_tensor_slices((
        (X_train, X_train_cat), y_train
    )).batch(config.batch_size).prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_tensor_slices((
        (X_val, X_val_cat), y_val
    )).batch(config.batch_size).prefetch(tf.data.AUTOTUNE)

    history = model.model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=epochs,
        callbacks=callbacks,
        verbose=0
    )

    if os.path.exists(checkpoint_path):
        model.model.load_weights(checkpoint_path)
        os.remove(checkpoint_path)
    
    val_loss = float(np.min(history.history["val_loss"]))
    y_pred = model.model.predict([X_test, X_test_cat], verbose=0).reshape(-1).astype(np_f32)
    training_time = time.time() - t0

    metrics = {
        "Model": name_model,
        "Val_loss": float(val_loss),
        "MSPE": float(mean_squared_error(y_test, y_pred)),
        "RMSE": float(np.sqrt(float(mean_squared_error(y_test, y_pred)))),
        "MAE": float(mean_absolute_error(y_test, y_pred)),
        "R2": float(r2_score(y_test, y_pred)),
        "Time": float(training_time),
    }
    
    del train_dataset, val_dataset
    gc.collect()
    
    return metrics, model


def cleanup_tf_session():
    K.clear_session()
    gc.collect()
    try:
        tf.keras.backend.clear_session()
    except:
        pass

### Experiment Setup

In [8]:
# Model Setup
seed = 1
epochs = 500
batch_size = 256
num_sample = 2500
huber_delta = 1.345
HP_DR = [0, 0.2, 0.5]

# Basis Setup
num_basis = [10**2, 19**2, 37**2]
knot_num = 1400
order_max = 1400

fixed_order = 100

# Repeat 
repeat_experiment = 10

In [9]:
# Initialize results storage for different dropout rates
all_dr_results = {}

for dr_idx, dropout_rate in enumerate(HP_DR):
    print(f"\n{'#'*80}")
    print(f"# DROPOUT RATE: {dropout_rate}")
    print(f"# Progress: {dr_idx+1}/{len(HP_DR)}")
    print(f"{'#'*80}\n")
    
    # Initialize experiment results for this dropout rate
    experiment_results = {
        model: {"MSPE": [], "RMSE": [], "MAE": [], "R2": []}
        for model in ["DK_W", "DK_W_new", "DK_mrts", "DK_S", "DK_S_H"]
    }
    
    for repeat in range(repeat_experiment):
        current_seed = seed + repeat
        
        print(f"\n{'='*80}")
        print(f"🏃 DR={dropout_rate} | Repeat {repeat+1}/{repeat_experiment} | Seed={current_seed}")
        print(f"{'='*80}")
        
        # Generate data
        dataframe = simulate_data(num_sample=num_sample, seed=current_seed)
        location_data, location_data_norm, categorical_data, y_combined = data_preprocessing(dataframe)
        

        # Compute max MRTS_Sphere
        max_Phi_sphere, idx_knot, knot = precompute_max_mrts("sphere", location_data, knot_num, order_max, knot=None)
        max_Phi_sphere = max_Phi_sphere.astype(dtype_basis, copy=False)
        Phi_sphere = max_Phi_sphere[:, :fixed_order].astype(np_f32)

        # Compute max MRTS_Euclidean
        max_Phi_mrts, idx_knot_mrts, knot_mrts = precompute_max_mrts("mrts", location_data, knot_num, order_max, knot=location_data[idx_knot])
        max_Phi_mrts = max_Phi_mrts.astype(dtype_basis, copy=False)
        Phi_mrts = max_Phi_mrts[:, :fixed_order].astype(np_f32)
        
        # Compute Wendland basis
        Phi_wendland = precompute_wendland(location_data_norm, num_basis)
    
        
        # Store metrics for this repeat
        Record = {}
        
        # 1. DeepKriging_wendland (DK_W) - fixed dropout=0.5
        print(f"\n--- Training DK_W (Wendland basis, DR=0.5) ---")
        parts = prepare_data(categorical_data, Phi_wendland, y_combined, current_seed)
        with strategy.scope():
            metric, model = train_eval(
                "DeepKriging_wendland", epochs, batch_size, "mse", 0.5, *parts
            )
        Record["DK_W"] = {
            "MSPE": metric["MSPE"], "RMSE": metric["RMSE"], 
            "MAE": metric["MAE"], "R2": metric["R2"],
            "Time": metric["Time"]
        }
        del model
        cleanup_tf_session()
        del parts
        gc.collect()
        
        
        # 2. DeepKriging_wendland_new (DK_W_new) - variable dropout
        print(f"\n--- Training DK_W_new (Wendland basis + ModelConfig, DR=0.5) ---")
        parts = prepare_data(categorical_data, Phi_wendland, y_combined, current_seed)
        with strategy.scope():
            metric, model = train_eval(
                "DeepKriging_wendland_new", epochs, batch_size, "mse", 0.5, *parts
            )
        Record["DK_W_new"] = {
            "MSPE": metric["MSPE"], "RMSE": metric["RMSE"], 
            "MAE": metric["MAE"], "R2": metric["R2"],
            "Time": metric["Time"]
        }
        del model
        cleanup_tf_session()
        del parts
        gc.collect()

        # 3. DeepKriging_mrts - variable dropout
        print(f"\n--- Training DK_mrts (Euclidean basis, DR={dropout_rate}) ---")
        parts = prepare_data(categorical_data, Phi_mrts, y_combined, current_seed)
        with strategy.scope():
            metric, model = train_eval(
                "DeepKriging_mrts", epochs, batch_size, "mse", dropout_rate, *parts
            )
        Record["DK_mrts"] = {
            "MSPE": metric["MSPE"], "RMSE": metric["RMSE"], 
            "MAE": metric["MAE"], "R2": metric["R2"],
            "Time": metric["Time"]
        }
        del model
        cleanup_tf_session()
        del parts
        gc.collect()
        
        # 4. DeepKriging_sphere (DK_S) - variable dropout
        print(f"\n--- Training DK_S (Sphere basis, DR={dropout_rate}) ---")
        parts = prepare_data(categorical_data, Phi_sphere, y_combined, current_seed)
        with strategy.scope():
            metric, model = train_eval(
                "DeepKriging_sphere", epochs, batch_size, "mse", dropout_rate, *parts
            )
        Record["DK_S"] = {
            "MSPE": metric["MSPE"], "RMSE": metric["RMSE"], 
            "MAE": metric["MAE"], "R2": metric["R2"],
            "Time": metric["Time"]
        }
        del model
        cleanup_tf_session()
        del parts
        gc.collect()
        
        # 5. DeepKriging_sphere_Huber (DK_S_H) - variable dropout
        print(f"\n--- Training DK_S_H (Sphere basis + Huber, DR={dropout_rate}) ---")
        parts = prepare_data(categorical_data, Phi_sphere, y_combined, current_seed)
        with strategy.scope():
            metric, model = train_eval(
                "DeepKriging_sphere_Huber", epochs, batch_size, Huber(delta=huber_delta), dropout_rate, *parts
            )
        Record["DK_S_H"] = {
            "MSPE": metric["MSPE"], "RMSE": metric["RMSE"], 
            "MAE": metric["MAE"], "R2": metric["R2"],
            "Time": metric["Time"]
        }
        del model
        cleanup_tf_session()
        del parts
        gc.collect()
        
        # Print current repeat results
        result_table = []
        for model_name in ["DK_W", "DK_W_new", "DK_mrts", "DK_S", "DK_S_H"]:
            result_table.append({
                "Model": model_name,
                "MSPE": f"{Record[model_name]['MSPE']:.4f}",
                "RMSE": f"{Record[model_name]['RMSE']:.4f}",
                "MAE": f"{Record[model_name]['MAE']:.4f}",
                "R2": f"{Record[model_name]['R2']:.4f}",
                "Time": f"{Record[model_name]['Time']:.2f}s"
            })
        
        df_res = pd.DataFrame(result_table)
        print("\n", df_res.to_markdown(index=False, tablefmt="github"), sep="")
        
        # Save results
        for model_name in ["DK_W", "DK_W_new", "DK_mrts", "DK_S", "DK_S_H"]:
            experiment_results[model_name]["MSPE"].append(Record[model_name]["MSPE"])
            experiment_results[model_name]["RMSE"].append(Record[model_name]["RMSE"])
            experiment_results[model_name]["MAE"].append(Record[model_name]["MAE"])
            experiment_results[model_name]["R2"].append(Record[model_name]["R2"])
        
        # Cleanup
        del Phi_wendland, Phi_sphere, max_Phi_sphere, dataframe
        del location_data, location_data_norm
        cleanup_tf_session()
        gc.collect()
        
        save_notebook()
        print(f"\n✅ Completed DR={dropout_rate}, Repeat {repeat+1}/{repeat_experiment}")
    
    # Store results for this dropout rate
    all_dr_results[dropout_rate] = experiment_results
    
    # Print summary for this dropout rate
    print(f"\n{'='*80}")
    print(f"📊 Summary for Dropout Rate = {dropout_rate}")
    print(f"{'='*80}")
    
    avg_results = []
    for model_name in ["DK_W", "DK_W_new", "DK_mrts", "DK_S", "DK_S_H"]:
        metrics = experiment_results[model_name]
        avg_results.append({
            "Model": model_name,
            "MSPE": f"{np.mean(metrics['MSPE']):.2f}±{np.std(metrics['MSPE']):.2f}",
            "RMSE": f"{np.mean(metrics['RMSE']):.2f}±{np.std(metrics['RMSE']):.2f}",
            "MAE": f"{np.mean(metrics['MAE']):.2f}±{np.std(metrics['MAE']):.2f}",
            "R2": f"{np.mean(metrics['R2']):.4f}±{np.std(metrics['R2']):.4f}",
        })
    
    df_avg = pd.DataFrame(avg_results)
    print("\n", df_avg.to_markdown(index=False, tablefmt="github"), sep="")
    print()

print(f"\n{'#'*80}")
print("# ALL EXPERIMENTS COMPLETED")
print(f"{'#'*80}\n")


################################################################################
# DROPOUT RATE: 0
# Progress: 1/3
################################################################################


🏃 DR=0 | Repeat 1/10 | Seed=1

Simulate Data | z mean: 12.9206 (1.5864), Variance: 6291.5864, Range: [-217.0852, 204.6014]


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.



--- Training DK_W (Wendland basis, DR=0.5) ---


I0000 00:00:1770171806.132962 1743691 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 17712 MB memory:  -> device: 0, name: NVIDIA RTX 4000 Ada Generation, pci bus id: 0000:70:00.0, compute capability: 8.9



📋 Model Config for DeepKriging_wendland
  activation          : 'relu'
  batch_size          : 256
  dropout_rate        : 0.5
  epochs              : 500
  hidden_units        : 100
  input_dim           : 1830
  loss                : 'mse'
  metrics             : ['mse', 'mae']
  num_hidden_layers   : 3
  optimizer           : Adam(...)
  output_type         : 'continuous'
  patience            : None
  verbose             : 0



...

💾 Notebook saved at 2026-02-04 12:37:19

✅ Completed DR=0.5, Repeat 8/10

🏃 DR=0.5 | Repeat 9/10 | Seed=9

Simulate Data | z mean: 11.8324 (1.6281), Variance: 6626.7510, Range: [-214.5135, 205.5588]

--- Training DK_W (Wendland basis, DR=0.5) ---

📋 Model Config for DeepKriging_wendland
  activation          : 'relu'
  batch_size          : 256
  dropout_rate        : 0.5
  epochs              : 500
  hidden_units        : 100
  input_dim           : 1830
  loss                : 'mse'
  metrics             : ['mse', 'mae']
  num_hidden_layers   : 3
  optimizer           : Adam(...)
  output_type         : 'continuous'
  patience            : None
  verbose             : 0


--- Training DK_W_new (Wendland basis + ModelConfig, DR=0.5) ---

📋 Model Config for DeepKriging_wendland_new
  activation          : 'relu'
  batch_size          : 256
  dropout_rate        : 0.5
  epochs              : 500
  hidden_layers       : [1024, 512, 256, 128, 64]
  input_dim           : 1830
  loss       

<IPython.core.display.Javascript object>

💾 Notebook saved at 2026-02-04 12:43:15

✅ Completed DR=0.5, Repeat 9/10

🏃 DR=0.5 | Repeat 10/10 | Seed=10

Simulate Data | z mean: 12.7194 (1.5889), Variance: 6311.7573, Range: [-209.6805, 206.5175]

--- Training DK_W (Wendland basis, DR=0.5) ---

📋 Model Config for DeepKriging_wendland
  activation          : 'relu'
  batch_size          : 256
  dropout_rate        : 0.5
  epochs              : 500
  hidden_units        : 100
  input_dim           : 1830
  loss                : 'mse'
  metrics             : ['mse', 'mae']
  num_hidden_layers   : 3
  optimizer           : Adam(...)
  output_type         : 'continuous'
  patience            : None
  verbose             : 0


--- Training DK_W_new (Wendland basis + ModelConfig, DR=0.5) ---

📋 Model Config for DeepKriging_wendland_new
  activation          : 'relu'
  batch_size          : 256
  dropout_rate        : 0.5
  epochs              : 500
  hidden_layers       : [1024, 512, 256, 128, 64]
  input_dim           : 1830
  loss     

<IPython.core.display.Javascript object>

💾 Notebook saved at 2026-02-04 12:49:49

✅ Completed DR=0.5, Repeat 10/10

📊 Summary for Dropout Rate = 0.5

| Model    | MSPE           | RMSE       | MAE        | R2            |
|----------|----------------|------------|------------|---------------|
| DK_W     | 3669.52±473.97 | 60.45±3.95 | 45.39±3.20 | 0.4280±0.0500 |
| DK_W_new | 3608.78±470.77 | 59.94±3.98 | 43.32±3.57 | 0.4375±0.0516 |
| DK_mrts  | 2339.61±408.68 | 48.18±4.28 | 38.17±3.56 | 0.6347±0.0574 |
| DK_S     | 112.01±22.99   | 10.52±1.11 | 6.87±0.55  | 0.9825±0.0035 |
| DK_S_H   | 127.06±29.89   | 11.19±1.40 | 6.97±0.51  | 0.9802±0.0043 |


################################################################################
# ALL EXPERIMENTS COMPLETED
################################################################################



In [10]:
print("\n" + "="*80)
print("📊 FINAL SUMMARY: All Dropout Rates")
print("="*80 + "\n")

# Create comprehensive summary table
summary_data = []

for dr in HP_DR:
    for model_name in ["DK_W", "DK_W_new", "DK_mrts", "DK_S", "DK_S_H"]:
        metrics = all_dr_results[dr][model_name]
        
        summary_data.append({
            "DR": f"{dr:.1f}",
            "Model": model_name,
            "MSPE (mean±std)": f"{np.mean(metrics['MSPE']):.2f}±{np.std(metrics['MSPE']):.2f}",
            "RMSE (mean±std)": f"{np.mean(metrics['RMSE']):.2f}±{np.std(metrics['RMSE']):.2f}",
            "MAE (mean±std)": f"{np.mean(metrics['MAE']):.2f}±{np.std(metrics['MAE']):.2f}",
            "R2 (mean±std)": f"{np.mean(metrics['R2']):.4f}±{np.std(metrics['R2']):.4f}",
        })

df_summary = pd.DataFrame(summary_data)
print(df_summary.to_markdown(index=False, tablefmt="github"))
print()

# Find best model for each metric
print("\n" + "="*80)
print("🏆 BEST MODELS (by RMSE)")
print("="*80 + "\n")

best_models = []
for dr in HP_DR:
    best_rmse = float('inf')
    best_model = None
    
    for model_name in ["DK_W", "DK_W_new", "DK_mrts", "DK_S", "DK_S_H"]:
        mean_rmse = np.mean(all_dr_results[dr][model_name]['RMSE'])
        if mean_rmse < best_rmse:
            best_rmse = mean_rmse
            best_model = model_name
    
    best_models.append({
        "Dropout Rate": f"{dr:.1f}",
        "Best Model": best_model,
        "RMSE": f"{best_rmse:.2f}±{np.std(all_dr_results[dr][best_model]['RMSE']):.2f}",
        "R2": f"{np.mean(all_dr_results[dr][best_model]['R2']):.4f}±{np.std(all_dr_results[dr][best_model]['R2']):.4f}"
    })

df_best = pd.DataFrame(best_models)
print(df_best.to_markdown(index=False, tablefmt="github"))
print()

# Model comparison across all dropout rates
print("\n" + "="*80)
print("📈 MODEL COMPARISON (averaged across all dropout rates)")
print("="*80 + "\n")

model_comparison = []
for model_name in ["DK_W", "DK_W_new", "DK_mrts", "DK_S", "DK_S_H"]:
    all_mspe = []
    all_rmse = []
    all_mae = []
    all_r2 = []
    
    for dr in HP_DR:
        all_mspe.extend(all_dr_results[dr][model_name]['MSPE'])
        all_rmse.extend(all_dr_results[dr][model_name]['RMSE'])
        all_mae.extend(all_dr_results[dr][model_name]['MAE'])
        all_r2.extend(all_dr_results[dr][model_name]['R2'])
    
    model_comparison.append({
        "Model": model_name,
        "MSPE": f"{np.mean(all_mspe):.2f}±{np.std(all_mspe):.2f}",
        "RMSE": f"{np.mean(all_rmse):.2f}±{np.std(all_rmse):.2f}",
        "MAE": f"{np.mean(all_mae):.2f}±{np.std(all_mae):.2f}",
        "R2": f"{np.mean(all_r2):.4f}±{np.std(all_r2):.4f}"
    })

df_comparison = pd.DataFrame(model_comparison)
print(df_comparison.to_markdown(index=False, tablefmt="github"))
print()

print("\n" + "="*80)
print("✅ ALL ANALYSIS COMPLETED")
print("="*80)


📊 FINAL SUMMARY: All Dropout Rates

|   DR | Model    | MSPE (mean±std)   | RMSE (mean±std)   | MAE (mean±std)   | R2 (mean±std)   |
|------|----------|-------------------|-------------------|------------------|-----------------|
|  0   | DK_W     | 3645.82±459.29    | 60.26±3.83        | 44.99±3.15       | 0.4316±0.0477   |
|  0   | DK_W_new | 3595.81±481.46    | 59.83±4.09        | 43.18±3.78       | 0.4392±0.0570   |
|  0   | DK_mrts  | 796.87±910.40     | 23.83±15.14       | 16.68±12.69      | 0.8729±0.1463   |
|  0   | DK_S     | 77.09±23.02       | 8.68±1.31         | 5.16±0.71        | 0.9879±0.0038   |
|  0   | DK_S_H   | 99.87±40.81       | 9.80±1.97         | 5.31±0.62        | 0.9845±0.0059   |
|  0.2 | DK_W     | 3678.08±446.79    | 60.53±3.71        | 45.19±2.99       | 0.4263±0.0484   |
|  0.2 | DK_W_new | 3647.48±451.55    | 60.28±3.75        | 43.55±3.29       | 0.4305±0.0553   |
|  0.2 | DK_mrts  | 2439.08±692.78    | 48.85±7.28        | 38.52±6.23       | 0.6238±0.08